# Course 4 - Applied Text Mining in Python
## Assignment 3: Spam Classification with Text Features

### Overview
This assignment explores text classification for spam detection using the SMS Spam Collection dataset. You will build progressively more sophisticated models using Count Vectorizer, TF-IDF Vectorizer, and Support Vector Machines, adding engineered features like document length, digit count, and non-word character count to improve AUC performance.

### Learning Objectives
- Vectorize text data using Count Vectorizer and TF-IDF Vectorizer
- Train Multinomial Naive Bayes classifiers on sparse text matrices
- Engineer numeric features from raw text (length, digit count, special characters)
- Combine sparse matrices with dense features using `scipy.sparse.hstack`
- Compare models using AUC (Area Under the ROC Curve)
- Apply character n-gram vectorization for robustness to spelling variations
- Extract and interpret model coefficients from Logistic Regression

### Dataset
- **File:** `assets/spam.csv`
- **Size:** 5,572 SMS messages (4,179 train / 1,393 test, `random_state=0`)
- **Target:** `target` column (1 = spam, 0 = ham/legitimate)
- **Imbalance:** ~13.4% spam

### Assignment Structure
| Question | Method | Key Parameters |
|----------|--------|---------------|
| Q1 | Data exploration | Percentage of spam in dataset |
| Q2 | CountVectorizer | Longest token in vocabulary |
| Q3 | CountVectorizer + MultinomialNB | alpha=0.1; report AUC |
| Q4 | TF-IDF + max tf-idf | 20 smallest and 20 largest max tf-idf features |
| Q5 | TF-IDF (min_df=3) + MultinomialNB | alpha=0.1; report AUC |
| Q6 | EDA | Average document length for spam vs non-spam |
| Q7 | TF-IDF (min_df=5) + doc length + SVC | C=10000; AUC using decision_function |
| Q8 | EDA | Average digit count for spam vs non-spam |
| Q9 | TF-IDF n-grams(1-3) + length + digits + LogReg | C=100, max_iter=1000; AUC |
| Q10 | EDA | Average non-word character count for spam vs non-spam |
| Q11 | Char n-grams(2-5) + 3 features + LogReg | First 2000 train samples; AUC + top/bottom 10 coefficients |

### Key Concept: The `add_feature` Helper
The provided `add_feature(X, feature_to_add)` function uses `scipy.sparse.hstack` to append dense feature columns to a sparse TF-IDF matrix, allowing text features and numeric features to be combined for training.

In [1]:
import pandas as pd
import numpy as np

spam_data = pd.read_csv('assets/spam.csv')

spam_data['target'] = np.where(spam_data['target']=='spam',1,0)
spam_data.head(10)

,text,target
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0
5,FreeMsg Hey there darling it's been 3 week's n...,1
6,Even my brother is not like to speak with me. ...,0
7,As per your request 'Melle Melle (Oru Minnamin...,0
8,WINNER!! As a valued network customer you have...,1
9,Had your mobile 11 months or more? U R entitle...,1


In [2]:
spam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5572 non-null   object
 1   target  5572 non-null   int32 
dtypes: int32(1), object(1)
memory usage: 65.4+ KB


In [3]:
from sklearn.model_selection import train_test_split
from icecream import ic


X_train, X_test, y_train, y_test = train_test_split(spam_data['text'], 
                                                    spam_data['target'], 
                                                    random_state=0)

ic(X_train.shape)

ic

|

X_train

.

shape

...

(

4179

,

)

(4179,)

### Question 1
What percentage of the documents in `spam_data` are spam?

*This function should return a float, the percent value (i.e. $ratio * 100$).*

In [4]:
def answer_one():
    
    import nltk
    
    # YOUR CODE HERE
    spam = spam_data['target'].sum()/spam_data['target'].count()

    #raise NotImplementedError()
    return spam*100 #Your answer here

answer_one()

13.406317300789663

### Question 2

Fit the training data `X_train` using a Count Vectorizer with default parameters.

What is the longest token in the vocabulary?

*This function should return a string.*

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

def answer_two():
    from sklearn.feature_extraction.text import CountVectorizer
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(X_train, y_train)
    features_list = vectorizer.get_feature_names_out()
    sorted_features_list = sorted(features_list, key = lambda val: len(val), reverse = True)
    print(sorted_features_list[0])
    # YOUR CODE HERE
    #raise NotImplementedError()
    return sorted_features_list[0] #Your answer here

answer_two()

com1win150ppmx3age16subscription


'com1win150ppmx3age16subscription'

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(X_train, y_train)
features_list = vectorizer.get_feature_names_out()
sorted_features_list = sorted(features_list, key = lambda val: len(val), reverse = True)
sorted_features_list[0]

'com1win150ppmx3age16subscription'

### Question 3

Fit and transform the training data `X_train` using a Count Vectorizer with default parameters.

Next, fit a fit a multinomial Naive Bayes classifier model with smoothing `alpha=0.1`. Find the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score

def answer_three():
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics import roc_auc_score
    import re

    #smoothing is a smoothing technique that helps tackle the problem of zero probability in the Naïve Bayes machine learning algorithm
    import nltk
    from sklearn.feature_extraction.text import CountVectorizer

    vectorizer = CountVectorizer()
    vectorizer.fit_transform(X_train)

    X_train_v = vectorizer.transform(X_train)
    X_test_v = vectorizer.transform(X_test)

    #print(X_train)
    model = MultinomialNB(alpha =0.1)
    model.fit(X_train_v, y_train)
    print('model score is {}', model.score(X_test_v, y_test))
    y_predicted = model.predict(X_test_v)
    print('model roc_auc score is {}', roc_auc_score(y_test, y_predicted))
    # YOUR CODE HERE
    #raise NotImplementedError()
    return roc_auc_score(y_test, y_predicted) #Your answer here

answer_three()

model score is {} 0.9921033740129217
model roc_auc score is {} 0.9720812182741116


0.9720812182741116

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score
import re

#smoothing is a smoothing technique that helps tackle the problem of zero probability in the Naïve Bayes machine learning algorithm
import nltk
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
vectorizer.fit_transform(X_train)


<4179x7354 sparse matrix of type '<class 'numpy.int64'>'
	with 55130 stored elements in Compressed Sparse Row format>

In [9]:

#print(X_train)
X_train_v = vectorizer.transform(X_train)
X_test_v = vectorizer.transform(X_test)

#print(X_train)
model = MultinomialNB(alpha=0.1)
model.fit(X_train_v, y_train)
print('model score is {}', model.score(X_test_v, y_test))
y_predicted = model.predict(X_test_v)
y_pp = model.predict_proba(X_test_v)

display('model roc_auc score is {}', roc_auc_score(y_test, y_predicted))
display('model roc_auc score is {}', roc_auc_score(y_test, y_pp[:,1]))

model score is {} 0.9921033740129217


'model roc_auc score is {}'

0.9720812182741116

'model roc_auc score is {}'

0.991545422134696

### Question 4

Fit and transform the training data `X_train` using a Tfidf Vectorizer with default parameters. The transformed data will be a compressed sparse row matrix where the number of rows is the number of documents in `X_train`, the number of columns is the number of features found by the vectorizer in each document, and each value in the sparse matrix is the tf-idf value. First find the **max** tf-idf value for every feature.

What 20 features have the smallest tf-idf and what 20 have the largest tf-idf among the **max** tf-idf values?

Put these features in two series where each series is sorted by tf-idf value. The index of the series should be the feature name, and the data should be the tf-idf.

The series of 20 features with smallest tf-idfs should be sorted smallest tfidf first, the list of 20 features with largest tf-idfs should be sorted largest first. Any entries with identical tf-ids should appear in lexigraphically increasing order by their feature name in boh series. For example, if the features "a", "b", "c" had the tf-idfs 1.0, 0.5, 1.0 in the series with the largest tf-idfs, then they should occur in the returned result in the order "a", "c", "b" with values 1.0, 1.0, 0.5.

*This function should return a tuple of two series
`(smallest tf-idfs series, largest tf-idfs series)`.*

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

def answer_four():
    # Fit TF-IDF on training data, then find max tf-idf value per feature (column)
    vectorizer = TfidfVectorizer()
    X_train_tfidf = vectorizer.fit_transform(X_train)
    
    # Max tf-idf value for each feature across all documents
    max_tfidf = X_train_tfidf.max(axis=0).toarray().flatten()
    feature_names = vectorizer.get_feature_names_out()
    
    # Create (feature_name, max_tfidf) pairs and sort
    feature_list = list(zip(feature_names, max_tfidf))
    
    # Sort ascending by (tfidf, name) for bottom 20
    sorted_asc = sorted(feature_list, key=lambda x: (x[1], x[0]))
    # Sort descending by tfidf then ascending by name for top 20
    sorted_desc = sorted(feature_list, key=lambda x: (-x[1], x[0]))
    
    bottom_20 = pd.Series([v for _, v in sorted_asc[:20]], 
                           index=[k for k, _ in sorted_asc[:20]])
    top_20 = pd.Series([v for _, v in sorted_desc[:20]], 
                        index=[k for k, _ in sorted_desc[:20]])
    
    return (bottom_20, top_20)

answer_four()

(aaniye          0.074475
 athletic        0.074475
 chef            0.074475
 companion       0.074475
 courageous      0.074475
 dependable      0.074475
 determined      0.074475
 exterminator    0.074475
 healer          0.074475
 listener        0.074475
 organizer       0.074475
 pest            0.074475
 psychiatrist    0.074475
 psychologist    0.074475
 pudunga         0.074475
 stylist         0.074475
 sympathetic     0.074475
 venaam          0.074475
 afternoons      0.091250
 approaching     0.091250
 dtype: float64,
 146tf150p    1.000000
 645          1.000000
 anything     1.000000
 anytime      1.000000
 beerage      1.000000
 done         1.000000
 er           1.000000
 havent       1.000000
 home         1.000000
 lei          1.000000
 nite         1.000000
 ok           1.000000
 okie         1.000000
 thank        1.000000
 thanx        1.000000
 too          1.000000
 where        1.000000
 yup          1.000000
 tick         0.980166
 blank        0.932702
 dt

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
vectorizer.fit(X_train)
result = vectorizer.transform(X_train)

max_feature_value = result.max(axis =0)
max_feature_val_array = max_feature_value.toarray()
ic(max_feature_val_array.shape)
max_feature_val_array_flattened =max_feature_val_array.reshape(-1)
ic(max_feature_val_array_flattened.shape)

ic

|

max_feature_val_array

.

shape

...

(

7354

,

)

(7354,)

In [12]:
max_feature_val_list=max_feature_val_array_flattened.tolist()

ic(type(max_feature_value))
ic(result.shape)
ic(max_feature_value.shape)
#ic(max_feature_val_list[0:10])
ic(type(max_feature_val_list))
ic(len(max_feature_val_list))
ic(max_feature_val_list[1])

ic

|

type

(

max_feature_value

...

1

]

:

0.36138462692384316

0.36138462692384316

In [13]:
idf = vectorizer.idf_
feature_names = vectorizer.get_feature_names_out()

result = vectorizer.fit_transform(X_train)

feature_list = list(zip( feature_names, max_feature_val_list))
ic(type(feature_list))
ic(feature_list[0])
#ic(feature_list[-20:])

sorted_list_descending = sorted(feature_list, key = lambda val: val[0])
sorted_list_descending = sorted(sorted_list_descending, key = lambda val: val[1], reverse = True)

sorted_list_ascending = sorted(feature_list, key = lambda val: (val[1],val[0]), reverse = False)

ic

|

type

(

feature_list

...

00

'

,

0.23731415804600853

)

In [14]:
#Sorting on the key in ascending order as tie breaker TBD
#sorted_list = sorted(feature_list, key = lambda val: val[0], ,  reverse = False)

top_20 = sorted_list_descending[0:20]
bottom_20=  sorted_list_ascending[0:20]

#display(top_20[0][0])
#display(top_20[0][1])
#display(bottom_20)

ic(top_20)
ic(bottom_20)
data_s1 = [  x[1] for x in top_20]
index_s1 =[  x[0] for x in top_20]

data_s2 = [  x[1] for x in bottom_20]
index_s2 =[  x[0] for x in bottom_20]

s1 = pd.Series(data=data_s1, index =index_s1 )

s2 = pd.Series(data=data_s2, index =index_s2 )
#s2 = s2.sort_values(ascending = True)

ic(s1)
ic(s2)

ic

|

top_20

:

[

...

0.091250

dtype

:

float64

aaniye          0.074475
athletic        0.074475
chef            0.074475
companion       0.074475
courageous      0.074475
dependable      0.074475
determined      0.074475
exterminator    0.074475
healer          0.074475
listener        0.074475
organizer       0.074475
pest            0.074475
psychiatrist    0.074475
psychologist    0.074475
pudunga         0.074475
stylist         0.074475
sympathetic     0.074475
venaam          0.074475
afternoons      0.091250
approaching     0.091250
dtype: float64

### Question 5

Fit and transform the training data `X_train` using a Tfidf Vectorizer ignoring terms that have a document frequency strictly lower than **3**.

Then fit a multinomial Naive Bayes classifier model with smoothing `alpha=0.1` and compute the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [15]:
def answer_five():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics import roc_auc_score

    # min_df=3 removes rare tokens that appear in fewer than 3 documents
    vectorizer = TfidfVectorizer(min_df=3)
    X_train_v = vectorizer.fit_transform(X_train)
    X_test_v = vectorizer.transform(X_test)

    clf = MultinomialNB(alpha=0.1)
    clf.fit(X_train_v, y_train)
    
    # Use predicted probabilities of class=1 (spam) for AUC
    y_scores = clf.predict_proba(X_test_v)[:, 1]
    return roc_auc_score(y_test, y_scores)

answer_five()

0.9954968337775665

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score
import re

#smoothing is a smoothing technique that helps tackle the problem of zero probability in the Naïve Bayes machine learning algorithm
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(min_df =3)
vectorizer.fit(X_train)
#vectorizer.fit_transform(X_train)

X_train_v = vectorizer.transform(X_train)
X_test_v = vectorizer.transform(X_test)

display(print(X_train_v.shape))
clf_NB = MultinomialNB(alpha = 0.1)

clf_NB.fit(X_train_v, y_train)

print('model score is {}', clf_NB.score(X_test_v, y_test))
y_predicted = clf_NB.predict(X_test_v)
p_pred_tuple = clf_NB.predict_proba(X_test_v)
p_predicted_lowp = p_pred_tuple[:,1]
p_predicted_highp = p_pred_tuple[:,0]

ic(p_pred_tuple[0:5])
ic(p_predicted_highp[0:5])
ic('model roc_auc score is {}', roc_auc_score(y_test, y_predicted))
ic ('model roc_auc score is {}', roc_auc_score(y_test, p_predicted_lowp)) #probability of test sample being in class 1
ic('model roc_auc score is {}', roc_auc_score(y_test, p_predicted_highp))



(4179, 2295)


None

ic

|

p_pred_tuple

...

)

:

0.004503166222433495

model score is {} 0.9834888729361091


('model roc_auc score is {}', 0.004503166222433495)

### Question 6

What is the average length of documents (number of characters) for not spam and spam documents?

*This function should return a tuple (average length not spam, average length spam).*

In [17]:
def answer_six():
    # Compare average document character length for non-spam (0) vs spam (1)
    spam_data['len_document'] = spam_data['text'].apply(len)
    avg_not_spam = spam_data[spam_data['target'] == 0]['len_document'].mean()
    avg_spam = spam_data[spam_data['target'] == 1]['len_document'].mean()
    return (avg_not_spam, avg_spam)

answer_six()

(71.02362694300518, 138.8661311914324)

In [18]:
display(spam_data.head(2))

spam_data['len_document'] = spam_data['text'].apply(len)
spam_grouped = spam_data.groupby(by = 'target').mean(numeric_only=True)
a=spam_grouped.loc[0,'len_document']
b=spam_grouped.loc[1,'len_document']


,text,target,len_document
0,"Go until jurong point, crazy.. Available only ...",0,111
1,Ok lar... Joking wif u oni...,0,29


<br>
<br>
The following function has been provided to help you combine new features into the training data:

In [19]:
def add_feature(X, feature_to_add):
    """
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    """
    from scipy.sparse import csr_matrix, hstack
    return hstack([X, csr_matrix(feature_to_add).T], 'csr')

### Question 7

Fit and transform the training data X_train using a Tfidf Vectorizer ignoring terms that have a document frequency strictly lower than **5**.

Using this document-term matrix and an additional feature, **the length of document (number of characters)**, fit a Support Vector Classification model with regularization `C=10000`. Then compute the area under the curve (AUC) score using the transformed test data.

*Hint: Since probability is set to false, use the model's `decision_function` on the test data when calculating the target scores to use in roc_auc_score*

*This function should return the AUC score as a float.*

In [20]:
from sklearn.svm import SVC

def answer_seven():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.svm import SVC
    from sklearn.metrics import roc_auc_score

    # TF-IDF features (min_df=5 removes very rare terms)
    vectorizer = TfidfVectorizer(min_df=5)
    X_train_v = vectorizer.fit_transform(X_train)
    X_test_v = vectorizer.transform(X_test)

    # Add document length as an additional feature
    X_train_len = X_train.apply(len)
    X_test_len = X_test.apply(len)
    X_train_v1 = add_feature(X_train_v, X_train_len)
    X_test_v1 = add_feature(X_test_v, X_test_len)

    # SVC with high C -> less regularization -> fits training data closely
    model = SVC(C=10000)
    model.fit(X_train_v1, y_train)

    # Use decision_function scores (not probabilities) for AUC since probability=False
    y_df = model.decision_function(X_test_v1)
    return roc_auc_score(y_test, y_df)

answer_seven()

0.9963202213809143

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

#TF-IDF Vectorizer is a measure of originality of a word by comparing the number of times a word appears in document with the number of documents the word appears in.
# Term frequency : in the particular doc
# 
vectorizer = TfidfVectorizer(min_df =5)
vectorizer.fit(X_train)

X_train_v = vectorizer.transform(X_train)
X_test_v = vectorizer.transform(X_test)

display(print(X_train_v.shape))
display(print(X_test_v.shape))
#clf_NB = MultinomialNB(alpha = 0.1)


ic(X_train[0])

X_train_len_document = X_train.apply(len)
X_test_len_document = X_test.apply(len)
ic(X_train_len_document[0])

X_train_v1 = add_feature(X_train_v, X_train_len_document)
X_test_v1 = add_feature(X_test_v, X_test_len_document)

display(print(X_train_v1.shape))
display(print(X_test_v1.shape))

from sklearn.svm  import SVC

model= SVC(C=10000)

model.fit(X_train_v1, y_train)
print('model score is {}', model.score(X_test_v1, y_test))
y_predicted = model.predict(X_test_v1)
y_df = model.decision_function(X_test_v1)

ic( print('model roc_auc score is {}', roc_auc_score(y_test, y_predicted)))
ic( print('model roc_auc score is {}', roc_auc_score(y_test, y_df)))


#X_train_1 = add_feature(X_train_v, spam_data['len_document'].values)

(4179, 1468)


None

(1393, 1468)


None

ic

...

)

)

:

None

model score is {} 0.9892318736539842
model roc_auc score is {} 0.9661689557407943
model roc_auc score is {} 0.9963202213809143


### Question 8

What is the average number of digits per document for not spam and spam documents?

*Hint: Use `\d` for digit class*

*This function should return a tuple (average # digits not spam, average # digits spam).*

In [22]:
def answer_eight():
    # Count digits (\d matches 0-9) per message; compare spam vs non-spam averages
    spam_data['digit_count'] = spam_data['text'].str.findall(r'\d').apply(len)
    avg_not_spam = spam_data[spam_data['target'] == 0]['digit_count'].mean()
    avg_spam = spam_data[spam_data['target'] == 1]['digit_count'].mean()
    return (avg_not_spam, avg_spam)

answer_eight()

(0.2992746113989637, 15.759036144578314)

In [23]:
spam_data['digit_count'] = spam_data['text'].str.findall(pat = r'\d').apply(len)
a=spam_data['digit_count'][spam_data['target']==1].mean()
b=spam_data['digit_count'][spam_data['target']==0].mean()

### Question 9

Fit and transform the training data `X_train` using a Tfidf Vectorizer ignoring terms that have a document frequency strictly lower than **5** and using **word n-grams from n=1 to n=3** (unigrams, bigrams, and trigrams).

Using this document-term matrix and the following additional features:
* the length of document (number of characters)
* **number of digits per document**

fit a Logistic Regression model with regularization `C=100` and `max_iter=1000`. Then compute the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
import re
from icecream import ic
import re

def answer_nine():

    # YOUR CODE HERE
    #from icecream import ic

    #Convert a collection of raw documents to a matrix of TF-IDF features.
    #Equivalent to CountVectorizer followed by TfidfTransformer.

    vectorizer = TfidfVectorizer(ngram_range = (1,3), min_df = 5)

    #Learn vocabulary and idf(inverse document frequency), return document-term matrix.
    #Returns Tf-idf-weighted document-term matrix.
    vectorizer.fit(X_train)
    X_train_v = vectorizer.transform(X_train)
    ic(X_train[0:2])
    X_train_len = [len(x) for x in X_train]
    ic(X_train_len[0:2])

    X_train_digits = [ len(re.findall(pattern = r'[\d]', string = x )) for x in X_train]
    #display(X_train_digits)
    #display(X_train_v.shape)

    X_train_v1 = add_feature(X_train_v, X_train_len)
    X_train_v2 = add_feature(X_train_v1, X_train_digits)
    #display(X_train_v2.shape)
    idf = vectorizer.idf_

    X_test_v = vectorizer.transform(X_test)
    ic(X_test_v)
    X_test_len = [len(x) for x in X_test]
    X_test_digits = [ len(re.findall(pattern = r'[\d]', string = x )) for x in X_test]


    X_test_v1 = add_feature(X_test_v, X_test_len)
    X_test_v2 = add_feature(X_test_v1, X_test_digits)
    display(X_test_v2.shape)


    LogRegClassifier = LogisticRegression(C=100, max_iter = 1000)
    LogRegClassifier.fit(X_train_v2, y_train)

    y_pred = LogRegClassifier.predict(X_test_v2)
    y_df = LogRegClassifier.decision_function(X_test_v2)
    y_pp = LogRegClassifier.predict_proba(X_test_v2)

    score_p = roc_auc_score(y_test, y_pred)
    score_pp = roc_auc_score(y_test, y_pp[:,1])
    score_df = roc_auc_score(y_test, y_df)

    ic(score_p)
    ic(score_pp)
    ic(score_df)

    #raise NotImplementedError()
    return roc_auc_score(y_test, y_pred)#Your answer here

answer_nine()

ic

|

X_train

[

0

...

|

score_df

:

0.9972836697621512

0.9704089774714361

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
import re
from icecream import ic

In [26]:
#Convert a collection of raw documents to a matrix of TF-IDF features.
#Equivalent to CountVectorizer followed by TfidfTransformer.

vectorizer = TfidfVectorizer(ngram_range = (1,3), min_df = 5)

#Learn vocabulary and idf(inverse document frequency), return document-term matrix.
#Returns Tf-idf-weighted document-term matrix.
X_train_v = vectorizer.fit_transform(X_train)
ic(X_train[0:1])

X_train_len = [len(x) for x in X_train]
ic(X_train_len[0:1])
X_train_digits = [ len(re.findall(pattern = r'[\d]+', string = x )) for x in X_train]
#display(X_train_digits)
#display(X_train_v.shape)

X_train_v1 = add_feature(X_train_v, X_train_len)
X_train_v2 = add_feature(X_train_v1, X_train_digits)
#display(X_train_v2.shape)
idf = vectorizer.idf_

X_test_v = vectorizer.transform(X_test)
ic(X_test_v)
X_test_len = [len(x) for x in X_test]
X_test_digits = [ len(re.findall(pattern = r'[\d]+', string = x )) for x in X_test]


X_test_v1 = add_feature(X_test_v, X_test_len)
X_test_v2 = add_feature(X_test_v1, X_test_digits)
display(X_test_v2.shape)


LogRegClassifier = LogisticRegression(C=100, max_iter = 1000)
LogRegClassifier.fit(X_train_v2, y_train)
y_pred = LogRegClassifier.predict(X_test_v2)
ic(roc_auc_score(y_test, y_pred))

#display(idf)
#display(vectorizer.vocabulary_)
#vectorizer.get_feature_names_out()[0:10]

ic

|

X_train

[

0

...

y_pred

)

:

0.9759031798040846

0.9759031798040846

### Question 10

What is the average number of non-word characters (anything other than a letter, digit or underscore) per document for not spam and spam documents?

*Hint: Use `\w` and `\W` character classes*

*This function should return a tuple (average # non-word characters not spam, average # non-word characters spam).*

In [27]:
def answer_ten():

    # YOUR CODE HERE
    mr = spam_data['text'].str.extractall(pat = r'([\W]+)')
    mr.reset_index(level =1, inplace = True)
    mr.index.name = 'index'
    mc_col = mr.groupby(level='index').agg(match_count = ('match','count'))

    spam_data['non_words_count'] = mc_col['match_count']

    avg_nw_spam_data = spam_data[spam_data['target']==1]['non_words_count'].mean()
    avg_nw_non_spam_data = spam_data[spam_data['target']==0]['non_words_count'].mean()
    ic(avg_nw_non_spam_data, avg_nw_spam_data)

    #raise NotImplementedError()
    return (avg_nw_non_spam_data, avg_nw_spam_data)#Your answer here

answer_ten()

ic

|

avg_nw_non_spam_data

:

14.383129025555787

avg_nw_spam_data

:

25.149933065595715

(14.383129025555787, 25.149933065595715)

In [28]:
ic(spam_data.shape)
#ic(spam_data.head)
# mr = spam_data['text'].str.extractall(pat = r'(\W)')
# mr.reset_index(level =1, inplace = True)
# mr.index.name = 'index'
# mc_col = mr.groupby(level='index').agg(match_count = ('match','count'))

# ic(mc_col.shape)
# ic(mc_col.head)

#spam_data['non_words_count'] = mc_col['match_count']

#avg_nw_spam_data = spam_data[spam_data['target']==1].mean()
#avg_nw_non_spam_data = spam_data[spam_data['target']==0].mean()

#ic(avg_nw_non_spam_data, avg_nw_spam_data)

spam_data['non_words_cnt'] = spam_data['text'].str.findall(pat = r'(\W)').apply(len)
a=spam_data[spam_data['target']==1]['non_words_cnt'].mean()
b= spam_data[spam_data['target']==0]['non_words_cnt'].mean()

ic(spam_data.head)
ic(a,b)




ic

|

spam_data

.

shape

...

,

b

:

17.29181347150259

(29.041499330655956, 17.29181347150259)

### Question 11

Fit and transform the **first 2000 rows** of training data X_train using a Count Vectorizer ignoring terms that have a document frequency strictly lower than **5** and using **character n-grams from n=2 to n=5.**

To tell Count Vectorizer to use character n-grams pass in `analyzer='char_wb'` which creates character n-grams only from text inside word boundaries. This should make the model more robust to spelling mistakes.

Using this document-term matrix and the following additional features:
* the length of document (number of characters)
* number of digits per document
* **number of non-word characters (anything other than a letter, digit or underscore.)**

fit a Logistic Regression model with regularization C=100 and max_iter=1000. Then compute the area under the curve (AUC) score using the transformed test data.

Also **find the 10 smallest and 10 largest coefficients from the model** and return them along with the AUC score in a tuple.

The list of 10 smallest coefficients should be sorted smallest first, the list of 10 largest coefficients should be sorted largest first.

The three features that were added to the document term matrix should have the following names should they appear in the list of coefficients:
['length_of_doc', 'digit_count', 'non_word_char_count']

*This function should return a tuple `(AUC score as a float, smallest coefs list, largest coefs list)`.*

In [29]:
def answer_eleven():
    import re
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split

    # Use only first 2000 training samples
    X_train_2000 = X_train[:2000]
    y_train_2000 = y_train[:2000]

    # Character n-gram vectorizer: char_wb creates n-grams only within word boundaries
    # This makes the model more robust to spelling variations
    vect = CountVectorizer(min_df=5, ngram_range=(2, 5), analyzer='char_wb')
    X_train_v = vect.fit_transform(X_train_2000)
    X_test_v = vect.transform(X_test)

    # Add three engineered numeric features
    X_train_len = X_train_2000.str.len()
    X_train_digits = X_train_2000.str.count(r'\d')
    X_train_nonword = X_train_2000.str.count(r'\W')

    X_test_len = X_test.str.len()
    X_test_digits = X_test.str.count(r'\d')
    X_test_nonword = X_test.str.count(r'\W')

    X_train_final = add_feature(add_feature(add_feature(X_train_v, X_train_len),
                                            X_train_digits), X_train_nonword)
    X_test_final = add_feature(add_feature(add_feature(X_test_v, X_test_len),
                                           X_test_digits), X_test_nonword)

    clf = LogisticRegression(C=100, max_iter=1000)
    clf.fit(X_train_final, y_train_2000)

    y_scores = clf.decision_function(X_test_final)
    auc = roc_auc_score(y_test, y_scores)

    # Build named feature list: vectorizer features + 3 added feature names
    feature_names = list(vect.get_feature_names_out()) + \
                    ['length_of_doc', 'digit_count', 'non_word_char_count']
    coefs = clf.coef_[0]

    # Sort by coefficient value; take bottom 10 (most ham-indicative) and top 10 (most spam-indicative)
    coef_series = pd.Series(coefs, index=feature_names).sort_values()
    smallest = list(coef_series[:10].index)
    largest = list(coef_series[-10:][::-1].index)

    return (auc, smallest, largest)

answer_eleven()

(0.9931964416073883,
 ['. ', '..', ' i', 'at', ' m', 'n ', 'he', 'i ', 'ok', '? '],
 ['digit_count', 'co', 'ne', 'xt', ' st', 'xt ', 's ', 'es', ' f', 'on'])

In [30]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

spam_data['non_word_chars'] = spam_data['text'].str.findall(pat = r'(\W)').apply(len)
ic(spam_data['non_word_chars'].sum())

spam_data['text'].iloc[3314]

ic

|

spam_data

[

'

...

(

)

:

105127

'FREE MESSAGE Activate your 500 FREE Text Messages by replying to this message with the word FREE For terms & conditions, visit www.07781482378.com'

In [31]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

spam_data['non_word_chars'] = spam_data['text'].str.findall(pat = r'(\W)').apply(len)
ic(spam_data['non_word_chars'].sum())

X_train_2000 = X_train[0:2000]

X_train_f3, X_test_f3, y_train_f3, y_test_f3= train_test_split(spam_data['non_word_chars'], 
                                                    spam_data['target'], 
                                                    random_state=0)

#ic(X_train_f3[0:10])
ic(X_train_f3.dtype)
y_test_f3.name = y_test.name
y_train_f3.name = y_test.name

#ic(y_test_f3[0:10], y_test[0:10])

if y_test_f3.values.all ==y_test.values.all:
    print (True)
if y_train_f3.values.all == y_train.values.all:
    print (True)

vectorizer = CountVectorizer(min_df=5, ngram_range=(2,5), analyzer='char_wb')

X_train_v = vectorizer.fit_transform(X_train_2000)

#X_train_len = [len(x) for x in X_train_2000]
#X_train_digits = [ len(re.findall(pattern = r'[\d]+', string = x )) for x in X_train_2000]

#ic(X_train_2000.shape)
#ic(type(X_train_2000))


X_train_2000_len_of_doc = X_train_2000.str.len()
X_train_2000.name ='length_of_doc'
X_train_2000_digit_count =X_train_2000.str.findall(pat = r'(\d)')
#ic(X_train_2000_digit_count[0:10])

X_train_2000_digit_count.name = 'digit_count'
#ic(X_train_2000['digit_count'].shape)

X_train_2000_digit_count=X_train_2000_digit_count.apply(len)
#ic(X_train_2000_digit_count[0:10])
ic(np.sum(X_train_2000_digit_count))


#ic(X_train_2000['digit_count'][0:10])

#display(X_train_digits)
#display(X_train_v.shape)


#X_train_f3 =X_train_f3.astype('str')

X_train_f3_2000 =X_train_f3[0:2000]

#X_test_f3 =X_test_f3.astype('str')
X_test_f3_2000 =X_test_f3[0:2000]

#ic(X_train_f3_2000[0:10])
#ic(X_test_f3_2000[0:10])

#ic(X_train_v[0:10])

ic(X_train_v.shape)

X_train_v1 = add_feature(X_train_v, X_train_2000_len_of_doc)
#ic(X_train_2000['length_of_doc'].shape)
ic(X_train_v1.shape)
#ic(X_train_2000['digit_count'].shape)

X_train_v2 = add_feature(X_train_v1, X_train_2000_digit_count)
ic(X_train_v2.shape)
X_train_v3 = add_feature(X_train_v2, X_train_f3[0:2000])
ic(X_train_v3.shape)

X_test_v = vectorizer.transform(X_test)
ic(X_test_v)

#X_test_len = [len(x) for x in X_test]
#X_test_digits = [ len(re.findall(pattern = r'[\d]+', string = x )) for x in X_test]

X_test_len_of_doc = X_test.str.len()
X_test_len_of_doc.name = 'length_of_doc'

X_test_digit_count =X_test.str.findall(pat = r'(\d)')
X_test_digit_count.name = 'digit_count'
X_test_digit_count=X_test_digit_count.astype('str').apply(len)

ic(X_test_v.shape)
X_test_v1 = add_feature(X_test_v, X_test_len_of_doc )
X_test_v2 = add_feature(X_test_v1, X_test_digit_count)
X_test_v3 = add_feature(X_test_v2, X_test_f3[0:2000])
ic(X_test_v3.shape)

feature_names = vectorizer.get_feature_names_out()

feature_coef_list = list(zip(feature_names, LogRegClassifier.coef_[0]))
ic(feature_coef_list)

display(print("sum of lengths:", sum(X_train_2000_len_of_doc), "sum of digits:", sum(X_train_2000_digit_count), " sum of word_chars:", sum(X_train_f3[0:2000])))

#ic(X_train_v3[0:10])
#new_feature_names = [x for x in vectorizer.get_feature_names_out() if x=='non_word_chars']
 
#ic(new_feature_names)
               
LogRegClassifier = LogisticRegression(C=100, max_iter = 1000)
LogRegClassifier.fit(X_train_v3, y_train[0:2000])
ic(LogRegClassifier.coef_.shape)
ic(LogRegClassifier.coef_[0][-5:])
ic(LogRegClassifier.coef_.sort(axis =1))
ic(LogRegClassifier.coef_)

min_coeff_list =LogRegClassifier.coef_[0, 0:10].tolist()
max_coeff_list = LogRegClassifier.coef_[0, -10:]
max_coeff_list_sorted = max_coeff_list[::-1].tolist()

#y_pp = LogRegClassifier.predict_proba(X_test_v3)
y_df = LogRegClassifier.decision_function(X_test_v3)
#y_pred = LogRegClassifier.predict(X_test_v3)
#score = roc_auc_score(y_test, y_pred)
#score = roc_auc_score(y_test, y_pp[:,1])
score = roc_auc_score(y_test, y_df)

ic(score)
ic(score, min_coeff_list, max_coeff_list_sorted)

ic

|

spam_data

[

'

...

0.708874795158627

,

0.7034061408316781

]

(0.9265869310561432,
 [-1.935903870223594,
  -1.897276540967369,
  -1.2975211754931462,
  -1.2496884358206133,
  -1.1587150417575063,
  -0.9909354114481588,
  -0.9803215316500122,
  -0.9450896788091971,
  -0.9203381143573733,
  -0.8502741932654625],
 [2.1998681292346323,
  1.2690032199562569,
  1.2193675505951527,
  1.0531444309052622,
  0.901053253655023,
  0.8991662997900626,
  0.7554020929679127,
  0.7403077330416599,
  0.708874795158627,
  0.7034061408316781])

In [32]:
# (exploratory debug cell - removed orphaned code)
pass